# 08 · From Data Preparation to Model Evaluation

Build a complete pipeline from data inspection to model evaluation and predictions for four new records. Write your solution in the empty code cells provided below each task.

Short runnable examples use small illustrative datasets and variables prefixed with `demo_`. Run them in order: they build on one another to demonstrate the complete workflow. Suggested tools and further hints appear with the assignments.

Complete the optional PCA block **before task 7**; do not change configurations after opening the test set. Once finished, use `Restart Kernel and Run All` to check that your solution runs in order.


## Environment setup and data loading

1. Open the notebook with `08-Data-preprocessing` as the working directory. If you use the repository root, prefix relative data paths with this folder. The README describes packages and setup.
2. Run the imports in the example below; NumPy uses the name `np` and pandas uses `pd`. Set `SEED` to **42** for every random split.
3. Load `data/machine-records.csv` into `raw` and display its first few rows. Keep the original table unchanged.

**Use:** `pd.read_csv`, `head()`, and `display` or `print`. Write short, separate statements.

<details><summary>Hint: imports</summary>

Use `from module import Name` to import an object. The tools belong to these modules:

- `sklearn.model_selection`: `train_test_split`; `sklearn.base`: `clone`.
- `sklearn.dummy`: `DummyClassifier`; `sklearn.neighbors`: `KNeighborsClassifier`.
- `sklearn.impute`: `SimpleImputer`; `sklearn.preprocessing`: `StandardScaler`, `OneHotEncoder`.
- `sklearn.compose`: `ColumnTransformer`; `sklearn.pipeline`: `Pipeline`.
- `sklearn.metrics`: `balanced_accuracy_score`, `ConfusionMatrixDisplay`; for the optional block, `sklearn.decomposition`: `PCA`.

For plots, import `matplotlib.pyplot` as `plt`. Jupyter provides `display`; it can also be imported from `IPython.display`. Pandas reads empty CSV fields as missing values.

</details>

### Example: Guided examples: separate miniature data


In [ ]:
from IPython.display import display  # Display tables in your own tasks.
import matplotlib.pyplot as plt  # Display the confusion-matrix plot.
from sklearn.metrics import ConfusionMatrixDisplay  # Compare actual and predicted classes.
import numpy as np  # Numerical operations and a missing-value marker for examples and independent work.
import pandas as pd  # Example tables are independent of the exercise files.
from sklearn.model_selection import train_test_split  # Split example inputs and targets together.
from sklearn.dummy import DummyClassifier  # A simple reference classifier.
from sklearn.base import clone  # Create a fresh unfitted copy of a model configuration.
from sklearn.impute import SimpleImputer  # Impute values using training data.
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # Scale numbers and encode categories.
from sklearn.pipeline import Pipeline  # Connect multiple steps into one pipeline.
from sklearn.compose import ColumnTransformer  # Apply different transformations to different columns.
from sklearn.neighbors import KNeighborsClassifier  # Classify using neighbouring training points.
from sklearn.metrics import balanced_accuracy_score  # Average the recalls of both classes.
from sklearn.decomposition import PCA  # Project numerical inputs onto principal components.
demo_data = pd.DataFrame({  # Twelve artificial independent samples, separate from the assignment data.
    "length": [1., 2., np.nan, 4., 5., 6., 7., 8., 9., 10., 11., 12.],  # The first numerical feature has one missing value.
    "mass": [12., 18., 35., 39., 52., 58., 71., 83., 87., np.nan, 108., 124.],  # The second numerical feature has a different scale and one missing value.
    "shape": ["round", "square"] * 6,  # One unordered categorical feature without missing values.
    "label": [0] * 6 + [1] * 6,  # Two equally sized classes solely for demonstrating syntax.
})  # End of the small table shared by all examples.
print(demo_data.head(3))  # A short preview of the examples' own data.


**Your code:**


## Problem and data

One row represents an **independent simulated machine situation**, not a neighbouring moment in a time series. Situations are independent for this exercise. The target `state` has two classes: **0 = normal simulated state**, **1 = simulated state requiring inspection**. This is not a physical fault model.

| Columns | Role in this exercise |
|---|---|
| `record_id` | situation identifier, used only to check duplicates |
| `temperature`, `vibration`, `pressure`, `power`, `speed`, `load` | six numerical features with different scales |
| `mode`, `environment` | operating mode and environment; unordered categories |
| `state` | target class, never a model input |

The CSV contains 720 unique situations and 6 duplicate records. Empty fields are read as `np.nan`. Data-collection documentation defines two corrections: `temperature = -999` marks a missing measurement, and `AUTO` means the same as `auto`. These fixed corrections and removal of identical duplicates precede splitting; they estimate nothing from data. Otherwise, a duplicated situation could appear in both training and test data.

We assume that all eight features are available **at prediction time**. Their availability would have to be verified in a real task. Statistical imputation, scaling, encoding and PCA are fitted only on training data. Test outliers are not removed based on model behaviour.

## 1. Inspection and known corrections

**Goal / output:** a table `clean` with 720 rows, no duplicated ID, no temperature value `-999` and no category `AUTO`. The summary `missing_counts` shows missing values **before** the fixed corrections; it does not estimate preprocessing parameters.

1. Fill in per-column missing counts and the number of repeated `record_id` values.
2. Remove repeated IDs, keeping the first record, and create a separate copy.
3. Replace the temperature sentinel with `np.nan` and standardize `AUTO` to `auto`.

**Use:** `isna()`, `sum()`, `duplicated(subset=...)`, `drop_duplicates(subset=..., keep="first")`, `copy()` and a simple `.loc[mask, "column"]` selection. Put each operation on its own line; no `apply`, `lambda` or `groupby` is needed.

<details><summary>Hint</summary>

`isna()` returns a table of yes/no values. Its `sum()` counts them by column. `duplicated` returns one yes/no value per row. A mask can have a name: `mask = table["column"] == value`; then `table.loc[mask, "column"] = replacement`. The corrections are supplied by the data documentation, not estimated from frequencies or quantiles.

</details>

**Think about:** Why is a duplicate of the same situation in training and test data a problem? Why are missing values not imputed with a median yet?

### Example: missing values, duplicates and fixed corrections


In [ ]:
demo_dirty = pd.concat([demo_data, demo_data.iloc[[0]]], ignore_index=True)  # Add one exact duplicate row to a separate scratch table.
demo_dirty.loc[1, "length"] = -1  # In this example, the documentation defines a negative length as missing.
demo_missing = demo_dirty.isna().sum()  # Count empty cells separately for each column.
demo_duplicates = demo_dirty.duplicated().sum()  # Count complete duplicate rows after their first occurrence.
demo_clean = demo_dirty.drop_duplicates().copy()  # Remove duplicates and make a separate table for corrections.
demo_bad_length = demo_clean["length"] < 0  # A mask selects only the documented invalid value.
demo_clean.loc[demo_bad_length, "length"] = np.nan  # The fixed correction estimates no statistic from the samples.
print(demo_missing, "duplicates:", demo_duplicates)  # Summary of original issues; later examples continue with demo_clean.


**Your code:**


## 2. Features and a fixed split

**Goal / output:** `X` contains eight features and `y` contains `state`. Training / validation / test contain 432 / 144 / 144 rows. Neither `record_id` nor `state` belongs in `X`.

1. Create `X` and `y`.
2. Hold out 20% for testing: development data `X_dev`, `y_dev` and test data `X_test`, `y_test`.
3. Hold out one quarter of the development 80% for validation. This leaves 60% of the original data for training. Name the results `X_train`, `X_val`, `y_train`, `y_val`.
4. Use `random_state=SEED` and stratification by the appropriate target in both splits. Keep the split unchanged between experiments.
5. Check that the parts do not overlap and that input indices match their target indices. Display row counts and class proportions in training and validation.

**Use:** `drop(columns=[...])`, target-column selection and `train_test_split(..., test_size=..., stratify=..., random_state=SEED)`. The function returns inputs for the first and second part, followed by targets for the first and second part.

<details><summary>Hint</summary>

The general order is `A_train, A_hold, b_train, b_hold = train_test_split(A, b, ...)`. The second split only receives development data, so `stratify` receives the development target. A quarter of 80% is 20% of the original dataset. Stratification approximately preserves class proportions; it does not make class sizes equal.

</details>

**Think about:** Why does the second split use `test_size=0.25` even though validation contains only 20% of all data?

### Example: split inputs and targets together


In [ ]:
demo_X = demo_clean.drop(columns="label")  # Inputs contain two numbers and one category, not the target class.
demo_y = demo_clean["label"]  # Keep the target separate from input features.
demo_X_dev, demo_X_test, demo_y_dev, demo_y_test = train_test_split(demo_X, demo_y, test_size=0.25, stratify=demo_y, random_state=7)  # Reserve three samples for testing before any fitting.
demo_X_train, demo_X_val, demo_y_train, demo_y_val = train_test_split(demo_X_dev, demo_y_dev, test_size=1/3, stratify=demo_y_dev, random_state=7)  # Three of nine development samples form validation; six remain for training.
print(len(demo_X_train), len(demo_X_val), len(demo_X_test))  # These small counts demonstrate syntax, not reliable performance estimation.


**Your code:**


## 3. Reference classifier

**Goal / output:** `baseline` predicts the most frequent training class. `baseline_score` is its validation **balanced accuracy**: the mean recall of both classes. The two classes therefore have equal weight in this metric even if their frequencies differ.

Create `DummyClassifier(strategy="most_frequent")`, fit it on training data only, predict validation data and calculate the metric. This classifier ignores the features, so it needs neither imputation nor encoding.

**Use:** the constructor, `.fit(inputs, target)`, `.predict(inputs)` and `balanced_accuracy_score(truth, predictions)`.

<details><summary>Hint</summary>

Fitting and predicting are two separate commands. `fit` receives both inputs and correct answers; `predict` receives only inputs. Validation `y` is used to calculate the metric, not to train the model. A constant prediction achieves balanced accuracy 0.5 when both classes are present.

</details>

**Think about:** Why can a constant prediction have high ordinary accuracy but balanced accuracy of only 0.5?

### Example: reference prediction and a metric


In [ ]:
demo_baseline = DummyClassifier(strategy="most_frequent")  # Configure a constant prediction of the most frequent training class.
demo_baseline.fit(demo_X_train, demo_y_train)  # This model uses only class frequencies and ignores features.
demo_baseline_pred = demo_baseline.predict(demo_X_val)  # Prediction receives inputs without validation answers.
print(balanced_accuracy_score(demo_y_val, demo_baseline_pred))  # True validation targets enter only the evaluation.


**Your code:**


## 4. Two preprocessing branches and a model

**Goal / output:** an unfitted `pipeline` that accepts the original eight columns and performs all preprocessing and classification. Nothing is fitted at this step yet.

Create `NUMERIC` containing `temperature`, `vibration`, `pressure`, `power`, `speed`, `load`, and `CATEGORICAL` containing `mode`, `environment`.

| Object | Required steps and their names |
|---|---|
| `numeric_pipe` | `impute`: `SimpleImputer(strategy="median")` → `scale`: `StandardScaler()` |
| `categorical_pipe` | `impute`: `SimpleImputer(strategy="most_frequent")` → `encode`: `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` |
| `prepare` | `ColumnTransformer` with a `num` branch for `NUMERIC` and a `cat` branch for `CATEGORICAL` |
| `pipeline` | `prepare`: the preceding transformer → `model`: `KNeighborsClassifier(n_neighbors=7)` |

Numerical features have different scales, while kNN compares distances. The categories in `mode` and `environment` have no natural order. One-hot encoding therefore creates a separate column for each known category. Here, `sparse_output=False` returns an ordinary dense matrix for this small dataset. With `handle_unknown="ignore"`, an unknown category at prediction time does not cause an error; its corresponding block contains only zeros.

**Use:** `Pipeline`, `ColumnTransformer`, the constructors in the table and the column lists you created. Keep the step names unchanged; subsequent tasks use them.

<details><summary>Hint</summary>

Pipeline syntax is `Pipeline([("step_name", object), ...])`. A branch in a `ColumnTransformer` has three parts: `("branch_name", transformer, column_list)`. Constructors only describe a procedure. Later, `fit` on the full pipeline estimates statistics and categories from training data and trains the model. There is no need to manually convert intermediate results back into pandas.

</details>

Documentation: [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

**Think about:** Why do numerical and categorical columns receive different steps? Why do we avoid using codes `auto=0`, `manual=1`, `eco=2` as numerical distances here?

### Example: numerical and categorical branches


In [ ]:
demo_numeric = Pipeline([  # Successive steps for the two numerical columns.
    ("fill", SimpleImputer(strategy="mean")),  # The mean will be learned during fit from training data only.
    ("scale", StandardScaler()),  # Scaling follows missing-value imputation.
])  # End of the numerical branch.
demo_prepare = ColumnTransformer([  # Each triple gives a name, a transformation and columns.
    ("numbers", demo_numeric, ["length", "mass"]),  # Both numerical columns use the same branch.
    ("shapes", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["shape"]),  # Categories are complete here, so this example needs no imputation before encoding.
])  # End of the column transformer.
demo_pipeline = Pipeline([  # The full pipeline will receive the three original input columns.
    ("inputs", demo_prepare),  # Preprocessing is fitted together with the model.
    ("classifier", KNeighborsClassifier(n_neighbors=1)),  # The miniature example uses one nearest neighbour.
])  # Construction has not yet estimated any parameters from data.


**Your code:**


## 5. Fit the full pipeline and validate

**Goal / output:** a fitted `fitted_pipeline`, its validation predictions `val_pred` and `val_score`.

Create an unfitted copy of the configuration with `clone`, fit the **whole** pipeline only on `X_train`, `y_train` and predict the original table `X_val`. Validation is not fitted or rescaled separately. Compare the result with the baseline using the same metric. Record the names `baseline` and `standard` and their validation scores in a small table; use `pd.DataFrame` with a dictionary of lists if helpful.

**Use:** `clone`, `.fit`, `.predict` and `balanced_accuracy_score`.

<details><summary>Hint</summary>

`copy = clone(configuration)` retains settings, not learned statistics. Call `fit` on this copy of the full pipeline. Its `predict` automatically applies the same imputation, scaling and categories as during training. Prediction therefore receives the eight original columns, not a manually transformed matrix.

</details>

Related reading: [data leakage and consistent preprocessing](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

**Think about:** How did the model compare with the baseline? Which pipeline components would leak information if fitted before splitting?

### Example: fitting and predicting an original table


In [ ]:
demo_fitted = clone(demo_pipeline)  # An unfitted copy preserves the configuration of every step.
demo_fitted.fit(demo_X_train, demo_y_train)  # Imputation, scaling, categories and model all learn from training data only.
demo_val_pred = demo_fitted.predict(demo_X_val)  # Validation uses the already fitted transformations without another fit.
demo_val_score = balanced_accuracy_score(demo_y_val, demo_val_pred)  # The same metric as for the reference model.
print(demo_val_score)  # This example score uses only three validation samples.


**Your code:**


## 6. One controlled experiment

**Goal / output:** `experiment` uses the same imputation, encoding, model and split, but **without scaling the numerical features**. `experiment_score` allows comparison with `val_score`.

Create another `clone(pipeline)`. The only change is replacing the scaling step with `"passthrough"`. Then fit on the same training data, predict the same validation data and compute the same metric. Keep the original fitted pipeline unchanged. Add the `unscaled` variant and its validation score to the comparison.

**Use:** `clone`, `.set_params(prepare__num__scale="passthrough")`, `.fit`, `.predict` and `balanced_accuracy_score`. Double underscores separate the names of nested steps.

<details><summary>Hint</summary>

`set_params` changes the configuration before fitting. `passthrough` means “skip this step”, not remove the numerical columns. The sequence is copy → one change → refit → validation. A sensible change is not guaranteed to improve the score; an unchanged or worse result is also a valid experiment.

</details>

Optionally, try a **different separate experiment**: in a fresh copy of the original pipeline, change only `prepare__num__impute__strategy="mean"`. Changing both imputation and scaling at once does not isolate either method's effect.

**Think about:** Before the experiment: what result do you expect and why? Afterwards: what does the difference in validation scores show, and what cannot be generalized from it?

### Example: a copy and one parameter change


In [ ]:
demo_experiment = clone(demo_pipeline)  # A separate configuration leaves the first model unchanged.
demo_experiment.set_params(classifier__n_neighbors=3)  # The double underscore selects a classifier-step parameter; only neighbour count changes.
demo_experiment.fit(demo_X_train, demo_y_train)  # Refit using the same training rows.
demo_experiment_pred = demo_experiment.predict(demo_X_val)  # The same validation rows permit comparison.
demo_experiment_score = balanced_accuracy_score(demo_y_val, demo_experiment_pred)  # Targets and metric do not change between experiments.
print(demo_val_score, demo_experiment_score)  # The difference concerns this single change on this small split.


**Your code:**


## Optional: PCA in the numerical branch

**Goal / output:** `pca_model` and `pca_score`. If you complete this block, add the `pca` variant to the validation comparison before the final test. Also display the sum of `explained_variance_ratio_` from the fitted PCA.

Create a copy of the original pipeline. Replace its numerical branch with **impute (median) → scale (StandardScaler) → pca (PCA with 3 components)**. Keep the categorical branch and kNN with seven neighbours unchanged. Refit the complete procedure on training data only and evaluate it on validation data.

**Use:** `Pipeline`, `PCA(n_components=3)`, `clone`, `.set_params(prepare__num=...)`, `.fit`, `.predict` and `balanced_accuracy_score`.

<details><summary>Hint</summary>

PCA receives imputed and scaled numbers, so it follows these two steps inside `num`, rather than following the resulting one-hot columns. Construction is the same as in task 4, with an extra step named `pca`. Three components replace the six numerical features; categorical columns remain. `explained_variance_ratio_` describes retained variance, not classification performance.

Retrieve the fitted numerical branch through `pca_model.named_steps["prepare"].named_transformers_["num"]` and store it in a variable. Select its `named_steps["pca"]` and sum that object's `explained_variance_ratio_`. Use the fitted branch: the column transformer fits copies of the original branch objects.

</details>

This experiment does not use the test set. If task 7 has already been completed, do not use the bonus to revise the selected model afterwards.

**Think about:** How did validation performance and the number of numerical model inputs change? Why is retained variance alone insufficient for choosing a classifier?

### Example: one component from two numbers


In [ ]:
demo_pca_numeric = Pipeline([  # A new numerical branch starts with the same preparation.
    ("fill", SimpleImputer(strategy="mean")),  # The same missing-number imputation as in the original variant.
    ("scale", StandardScaler()),  # The same scaling before projection.
    ("reduce", PCA(n_components=1)),  # Project two numerical features onto one principal component.
])  # Categorical indicators remain outside PCA.
demo_pca_model = clone(demo_pipeline)  # A separate copy of the original configuration.
demo_pca_model.set_params(inputs__numbers=demo_pca_numeric)  # Replace only the numerical branch; other steps stay the same.
demo_pca_model.fit(demo_X_train, demo_y_train)  # The PCA direction is also estimated from training data only.
demo_pca_pred = demo_pca_model.predict(demo_X_val)  # The same validation data as for previous variants.
demo_pca_score = balanced_accuracy_score(demo_y_val, demo_pca_pred)  # A metric evaluates classification quality, not retained variance alone.
print(demo_pca_score)  # Compare the result before using the test set.


**Your code:**


## 7. Finalize selection and evaluate the test set

**Goal / output:** a selected name `chosen_name`, a newly fitted `final_pipeline`, a single set of test predictions `test_pred`, and their `test_score` with a confusion matrix.

Use the validation-score table you created in the preceding tasks. From `standard` and `unscaled`, and optionally the previously completed `pca`, choose the **highest unrounded validation balanced accuracy**. Break ties by preferring `standard`, then `unscaled`, then `pca`. This is a simple rule for this exercise, not proof of a universally best method.

Store the selected name in `chosen_name` and its object in `selected_pipeline`: `standard` refers to `fitted_pipeline`, `unscaled` to `experiment`, and optional `pca` to `pca_model`.

**Use:** the objects from your preceding computations, `pd.concat([part1, part2])`, `clone`, `.fit`, `.predict`, `balanced_accuracy_score`, and `ConfusionMatrixDisplay.from_predictions` for the confusion matrix.

<details><summary>Hint</summary>

`chosen_name` is one of the names in the table, supplied as a text string. Do not reuse the selected object with its old statistics: `clone` creates a fresh configuration. Concatenate training and validation separately for `X` and `y`, keeping the same order. Fit the entire new pipeline on these 576 situations. Only then make a single prediction for `X_test`; its target is used only for evaluation.

</details>

**After viewing the test result, only interpretation and prediction of new data remain.** Choosing variants, thresholds or rows to remove based on test results would turn the test set into another validation set. Rerunning a fixed completed solution for reproducibility is not a new experiment.

**Think about:** How many class-1 situations were labelled 0? How do test and validation scores differ, and why should this difference not trigger further tuning on the test set?

### Example: validation selection and refitting


In [ ]:
demo_candidates = {"one": demo_fitted, "three": demo_experiment}  # The two main configurations; the optional PCA example is not required.
demo_scores = {"one": demo_val_score, "three": demo_experiment_score}  # Selection uses unrounded validation values.
demo_choice = max(demo_scores, key=demo_scores.get)  # Highest score; a tie retains the first inserted name.
demo_final = clone(demo_candidates[demo_choice])  # The chosen configuration starts without previously learned statistics.
demo_final.fit(demo_X_dev, demo_y_dev)  # Refit the complete pipeline on nine development samples.
demo_test_pred = demo_final.predict(demo_X_test)  # Predict the test set only after selection is finalized.
demo_test_score = balanced_accuracy_score(demo_y_test, demo_test_pred)  # After splitting, test targets are used only for this final evaluation.
print(demo_choice, demo_test_score)  # Three test samples only demonstrate the procedure; the result is not used for tuning.


**Your code:**


## 8. Four new records

**Goal / output:** four predictions `new_pred` from the already fitted `final_pipeline`. New data have no `state` column, so their accuracy cannot be calculated.

Load `data/new-records.csv` into `new_data`. It uses the same input schema and contains four new situations. One uses the previously unknown mode `service`. Select the same input columns as in `X` and predict them using the full final pipeline. **No further `fit` and no rebuilding categories.** Display `record_id` alongside the corresponding predictions; IDs serve only as labels.

**Use:** selection `new_data[X.columns]` and `.predict` on the final pipeline.

<details><summary>Hint</summary>

The model still receives the original table, not manually encoded columns. The encoder maps the unknown mode to zeros in the `mode` block. This does not mean the model has learned what `service` means or that its prediction must be reliable. The predictions are simulated outputs for subsequent assessment.

</details>

**Think about:** Why did the unknown category not cause an error? What else would be needed to assess whether these four predictions are correct?

### Example: new samples without further fitting


In [ ]:
demo_new = pd.DataFrame({"length": [2.5, 10.5], "mass": [26., np.nan], "shape": ["round", "oval"]})  # Two new samples with the same inputs; oval is an unknown category.
demo_new_pred = demo_final.predict(demo_new)  # The fitted pipeline imputes and encodes without another fit.
print(demo_new_pred)  # Without true targets, predictions can be shown but their accuracy cannot be evaluated.


**Your code:**
